<a href="https://colab.research.google.com/github/khushityaagi/Brain_Age_prediction/blob/main/Brain_Age_Prediction_Final.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
!pip install nibabel pandas scikit-learn scipy matplotlib opencv-python torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121

Looking in indexes: https://download.pytorch.org/whl/cu121


In [27]:
from google.colab import files
print(" Upload kaggle.json (Kaggle -> Account -> Create New API Token)")
files.upload()

!mkdir -p ~/.kaggle && mv "kaggle (1).json" ~/.kaggle/kaggle.json && chmod 600 ~/.kaggle/kaggle.json

 Upload kaggle.json (Kaggle -> Account -> Create New API Token)


Saving kaggle (1).json to kaggle (1) (1).json


In [28]:
!kaggle datasets list | head -n 10

ref                                                       title                                                size  lastUpdated                 downloadCount  voteCount  usabilityRating  
--------------------------------------------------------  ---------------------------------------------  ----------  --------------------------  -------------  ---------  ---------------  
ahmadrazakashif/bmw-worldwide-sales-records-20102024      BMW Worldwide Sales Records (2010–2024)            853348  2025-09-20 14:39:45.280000           5654        121  1.0              
grandmaster07/student-exam-score-dataset-analysis         Student exam score dataset analysis                  2430  2025-09-26 07:44:12.677000           2190         49  1.0              
anassarfraz13/student-success-factors-and-insights        Student Success: Factors & Insights                 96178  2025-09-24 07:58:55.117000           2730         54  1.0              
saadaliyaseen/analyzing-student-academic-trends        

In [29]:
!mkdir -p /content/data
!kaggle datasets download -d hamedamin/preprocessed-oasis-and-epilepsy-and-ixi -p /content/data

Dataset URL: https://www.kaggle.com/datasets/hamedamin/preprocessed-oasis-and-epilepsy-and-ixi
License(s): CC-BY-SA-3.0
100% 4.19G/4.21G [02:38<00:00, 52.3MB/s]
100% 4.21G/4.21G [02:38<00:00, 28.5MB/s]


In [30]:

!unzip -q /content/data/preprocessed-oasis-and-epilepsy-and-ixi.zip -d /content/data/

!apt-get -y install -qq unrar >/dev/null

!unrar x -y /content/data/a.rar /content/data/extracted/ || true
!unrar x -y /content/data/b.rar /content/data/extracted/ || true
!unrar x -y /content/data/c.rar /content/data/extracted/ || true

!find /content/data/extracted -maxdepth 3 -type f | head -n 20

unzip:  cannot find or open /content/data/preprocessed-oasis-and-epilepsy-and-ixi.zip, /content/data/preprocessed-oasis-and-epilepsy-and-ixi.zip.zip or /content/data/preprocessed-oasis-and-epilepsy-and-ixi.zip.ZIP.

UNRAR 6.11 beta 1 freeware      Copyright (c) 1993-2022 Alexander Roshal

Cannot open /content/data/a.rar
No such file or directory
No files to extract

UNRAR 6.11 beta 1 freeware      Copyright (c) 1993-2022 Alexander Roshal

Cannot open /content/data/b.rar
No such file or directory
No files to extract

UNRAR 6.11 beta 1 freeware      Copyright (c) 1993-2022 Alexander Roshal

Cannot open /content/data/c.rar
No such file or directory
No files to extract
find: ‘/content/data/extracted’: No such file or directory


In [31]:
from google.colab import files
print(" Upload your IXI.xls file now")
uploaded = files.upload()

 Upload your IXI.xls file now


Saving IXI.xls to IXI (1).xls


In [32]:
import pandas as pd
import re

meta = pd.read_excel('/content/IXI.xls')
meta.columns = [c.strip() for c in meta.columns]

id_col = [c for c in meta.columns if 'ixi' in c.lower() or 'id' == c.lower()][0]
age_col = [c for c in meta.columns if 'age' in c.lower()][0]

meta['SubjectID'] = meta[id_col].apply(lambda x: f"IXI{int(x):03d}" if pd.notnull(x) else None)
meta['Age'] = pd.to_numeric(meta[age_col], errors='coerce')

meta = meta[['SubjectID', 'Age']].dropna().drop_duplicates('SubjectID')

print(" Real IXI metadata loaded successfully:", meta.shape)
print(meta.head())

 Real IXI metadata loaded successfully: (565, 2)
  SubjectID        Age
1    IXI002  35.800137
2    IXI012  38.781656
3    IXI013  46.710472
4    IXI014  34.236824
5    IXI015  24.284736


In [34]:
import os
from glob import glob
import pandas as pd

data_root = '/content/data/'

files = glob(os.path.join(data_root, '', '.nii'), recursive=True)

file_df = pd.DataFrame({'image_path': [str(f) for f in files]})

file_df['SubjectID'] = file_df['image_path'].astype(str).str.extract(r'(IXI\d{3})')

merged = pd.merge(meta, file_df, on='SubjectID', how='inner')

print(" Merged metadata + MRI paths successfully:", merged.shape)
print(merged.head())

 Merged metadata + MRI paths successfully: (0, 3)
Empty DataFrame
Columns: [SubjectID, Age, image_path]
Index: []


In [36]:
import os, re, pandas as pd
from glob import glob

roots = ['/content/data', '/content/data/extracted', '/content']
exts  = ('.nii', '.nii.gz', '.png', '.jpg', '.jpeg', '.bmp', '.tif', '.tiff')

all_files = []
for rt in roots:
    for root, _, files in os.walk(rt):
        for f in files:
            if f.lower().endswith(exts) and 'ixi' in f.lower():
                all_files.append(os.path.join(root, f))

print(f"Found {len(all_files)} files containing IXI and supported extensions.")
for p in all_files[:10]:
    print(" sample:", p)

file_df = pd.DataFrame({'image_path': [str(f) for f in all_files]})
file_df['SubjectID_raw'] = file_df['image_path'].astype(str).str.extract(r'(IXI\d+)', flags=re.IGNORECASE)

def normalize_id(s):
    if pd.isna(s): return None
    m = re.search(r'ixi(\d+)', str(s), re.IGNORECASE)
    if not m: return None
    return f"IXI{int(m.group(1)):03d}"

file_df['SubjectID'] = file_df['SubjectID_raw'].apply(normalize_id)
file_df = file_df.dropna(subset=['SubjectID']).drop_duplicates('image_path')

meta = meta.copy()
meta['SubjectID'] = meta['SubjectID'].apply(normalize_id)
meta = meta.dropna(subset=['SubjectID', 'Age']).drop_duplicates('SubjectID')

merged = pd.merge(meta, file_df[['SubjectID','image_path']], on='SubjectID', how='inner')

print(f"Merged successfully: {merged.shape}")
display(merged.head(10))

if merged.empty:
    os.system("find /content/data -maxdepth 3 -type d | sed -n '1,50p'")
    os.system("find /content/data -maxdepth 4 -type f | sed -n '1,50p'")

Found 3600 files containing IXI and supported extensions.
 sample: /content/data/ixi_slices/IXI420/IXI420synthetic024.png
 sample: /content/data/ixi_slices/IXI420/IXI420synthetic019.png
 sample: /content/data/ixi_slices/IXI420/IXI420synthetic028.png
 sample: /content/data/ixi_slices/IXI420/IXI420synthetic005.png
 sample: /content/data/ixi_slices/IXI420/IXI420synthetic002.png
 sample: /content/data/ixi_slices/IXI420/IXI420synthetic001.png
 sample: /content/data/ixi_slices/IXI420/IXI420synthetic015.png
 sample: /content/data/ixi_slices/IXI420/IXI420synthetic007.png
 sample: /content/data/ixi_slices/IXI420/IXI420synthetic012.png
 sample: /content/data/ixi_slices/IXI420/IXI420synthetic026.png
Merged successfully: (1800, 3)


,SubjectID,Age,image_path
0,IXI013,46.710472,/content/data/ixi_slices/IXI013/IXI013syntheti...
1,IXI013,46.710472,/content/data/ixi_slices/IXI013/IXI013syntheti...
2,IXI013,46.710472,/content/data/ixi_slices/IXI013/IXI013syntheti...
3,IXI013,46.710472,/content/data/ixi_slices/IXI013/IXI013syntheti...
4,IXI013,46.710472,/content/data/ixi_slices/IXI013/IXI013syntheti...
5,IXI013,46.710472,/content/data/ixi_slices/IXI013/IXI013syntheti...
6,IXI013,46.710472,/content/data/ixi_slices/IXI013/IXI013syntheti...
7,IXI013,46.710472,/content/data/ixi_slices/IXI013/IXI013syntheti...
8,IXI013,46.710472,/content/data/ixi_slices/IXI013/IXI013syntheti...
9,IXI013,46.710472,/content/data/ixi_slices/IXI013/IXI013syntheti...


from matplotlib import pyplot as plt
_df_0['index'].plot(kind='hist', bins=20, title='index')
plt.gca().spines[['top', 'right',]].set_visible(False)

from matplotlib import pyplot as plt
import seaborn as sns
def _plot_series(series, series_name, series_index=0):
  palette = list(sns.palettes.mpl_palette('Dark2'))
  counted = (series['index']
                .value_counts()
              .reset_index(name='counts')
              .rename({'index': 'index'}, axis=1)
              .sort_values('index', ascending=True))
  xs = counted['index']
  ys = counted['counts']
  plt.plot(xs, ys, label=series_name, color=palette[series_index % len(palette)])

fig, ax = plt.subplots(figsize=(10, 5.2), layout='constrained')
df_sorted = _df_1.sort_values('index', ascending=True)
_plot_series(df_sorted, '')
sns.despine(fig=fig, ax=ax)
plt.xlabel('index')
_ = plt.ylabel('count()')

from matplotlib import pyplot as plt
import seaborn as sns
def _plot_series(series, series_name, series_index=0):
  palette = list(sns.palettes.mpl_palette('Dark2'))
  counted = (series['Age']
                .value_counts()
              .reset_index(name='counts')
              .rename({'index': 'Age'}, axis=1)
              .sort_values('Age', ascending=True))
  xs = counted['Age']
  ys = counted['counts']
  plt.plot(xs, ys, label=series_name, color=palette[series_index % len(palette)])

fig, ax = plt.subplots(figsize=(10, 5.2), layout='constrained')
df_sorted = _df_2.sort_values('Age', ascending=True)
_plot_series(df_sorted, '')
sns.despine(fig=fig, ax=ax)
plt.xlabel('Age')
_ = plt.ylabel('count()')

from matplotlib import pyplot as plt
_df_3['index'].plot(kind='line', figsize=(8, 4), title='index')
plt.gca().spines[['top', 'right']].set_visible(False)

In [37]:
import pandas as pd
lab = merged[['image_path','Age','SubjectID']].rename(columns={'Age':'age'})
lab = lab.dropna().drop_duplicates()
lab.to_csv('/content/labels.csv', index=False)
len(lab), lab.head(2)

(1800,
                                           image_path        age SubjectID
 0  /content/data/ixi_slices/IXI013/IXI013syntheti...  46.710472    IXI013
 1  /content/data/ixi_slices/IXI013/IXI013syntheti...  46.710472    IXI013)

In [38]:
from sklearn.model_selection import train_test_split
subs = sorted(lab['SubjectID'].unique().tolist())
train_subs, tmp = train_test_split(subs, test_size=0.30, random_state=42)
val_subs, test_subs = train_test_split(tmp, test_size=0.50, random_state=42)
train_df = lab[lab['SubjectID'].isin(train_subs)].reset_index(drop=True)
val_df   = lab[lab['SubjectID'].isin(val_subs)].reset_index(drop=True)
test_df  = lab[lab['SubjectID'].isin(test_subs)].reset_index(drop=True)
(len(train_subs), len(val_subs), len(test_subs), train_df.shape, val_df.shape, test_df.shape)

(42, 9, 9, (1260, 3), (270, 3), (270, 3))

In [51]:
from google.colab import drive
drive.mount('/content/drive')

project_path = '/content/drive/MyDrive/BrainAgeProject/'

Mounted at /content/drive


In [57]:
!pip install -q kaggle
from google.colab import files
u = files.upload()

import os, shutil
os.makedirs('/root/.kaggle', exist_ok=True)
src = next(iter(u.keys()))
shutil.move(src, '/root/.kaggle/kaggle.json')
os.chmod('/root/.kaggle/kaggle.json', 0o600)
!kaggle datasets list | head -n 10

Saving kaggle (1).json to kaggle (1) (2).json
ref                                                       title                                                size  lastUpdated                 downloadCount  voteCount  usabilityRating  
--------------------------------------------------------  ---------------------------------------------  ----------  --------------------------  -------------  ---------  ---------------  
ahmadrazakashif/bmw-worldwide-sales-records-20102024      BMW Worldwide Sales Records (2010–2024)            853348  2025-09-20 14:39:45.280000           5673        121  1.0              
grandmaster07/student-exam-score-dataset-analysis         Student exam score dataset analysis                  2430  2025-09-26 07:44:12.677000           2201         49  1.0              
anassarfraz13/student-success-factors-and-insights        Student Success: Factors & Insights                 96178  2025-09-24 07:58:55.117000           2736         54  1.0              
saadaliya

In [58]:
!kaggle datasets download -d hamedamin/preprocessed-oasis-and-epilepsy-and-ixi -p /content
!unzip -q /content/preprocessed-oasis-and-epilepsy-and-ixi.zip -d /content/mri_data_raw || echo "⚠ Zip missing or corrupt"
!apt-get -y install -qq unrar
!unrar x -y "/content/mri_data_raw/a.rar" /content/mri_data/
!ls -R /content/mri_data | head -n 40

Dataset URL: https://www.kaggle.com/datasets/hamedamin/preprocessed-oasis-and-epilepsy-and-ixi
License(s): CC-BY-SA-3.0
100% 4.20G/4.21G [01:38<00:00, 39.1MB/s]
100% 4.21G/4.21G [01:38<00:00, 46.0MB/s]

UNRAR 6.11 beta 1 freeware      Copyright (c) 1993-2022 Alexander Roshal

/content/mri_data_raw/a.rar is not RAR archive
No files to extract
ls: cannot access '/content/mri_data': No such file or directory


In [59]:
!ls -R /content/mri_data_raw | head -n 40

/content/mri_data_raw:
a.rar
mri_IXI_480_2
mr_IXI_480i

/content/mri_data_raw/mri_IXI_480_2:
rh.ppIXI306-IOP-0867-SAGFSPGR_-sIXI30_-0003-00001-000001-01.nii
rh.ppIXI308-Guys-0884-MPRAGESEN_-s302_-0301-00003-000001-01.nii
rh.ppIXI313-HH-2241-MADisoTFE1_-s3t219_-0301-00003-000001-01.nii
rh.pptnIXI306-IOP-0867-SAGFSPGR_-sIXI30_-0003-00001-000001-01.nii
rh.pptnIXI308-Guys-0884-MPRAGESEN_-s302_-0301-00003-000001-01.nii
rh.pptnIXI313-HH-2241-MADisoTFE1_-s3t219_-0301-00003-000001-01.nii
wj_IXI115-Guys-0738-T1.nii
wj_IXI116-Guys-0739-T1.nii
wj_IXI117-Guys-0763-T1.nii
wj_IXI118-Guys-0764-T1.nii
wj_IXI119-Guys-0765-T1.nii
wj_IXI120-Guys-0766-T1.nii
wj_IXI121-Guys-0772-T1.nii
wj_IXI122-Guys-0773-T1.nii
wj_IXI123-Guys-0774-T1.nii
wj_IXI126-HH-1437-T1.nii
wj_IXI127-HH-1451-T1.nii
wj_IXI128-HH-1470-T1.nii
wj_IXI129-Guys-0775-T1.nii
wj_IXI130-HH-1528-T1.nii
wj_IXI131-HH-1527-T1.nii
wj_IXI132-HH-1415-T1.nii
wj_IXI134-Guys-0780-T1.nii
wj_IXI135-Guys-0779-T1.nii
wj_IXI136-HH-1452-T1.nii
wj_IXI137-HH-147

In [61]:
import os
import numpy as np
import nibabel as nib
import tensorflow as tf
from tensorflow.keras import layers, models, optimizers
from sklearn.model_selection import train_test_split

DATA_DIR = "/content/mri_data_raw"

mri_files = []
for root, _, files in os.walk(DATA_DIR):
    for file in files:
        if file.endswith(".nii"):
            mri_files.append(os.path.join(root, file))

print("Total MRI files found:", len(mri_files))

def load_mri(path, target_shape=(64, 64, 64)):
    img = nib.load(path).get_fdata()
    img = tf.image.resize(tf.convert_to_tensor(img), target_shape[:2])
    img = img.numpy()
    img = img / np.max(img)
    return np.expand_dims(img, -1)

X = np.array([load_mri(f) for f in mri_files[:20]])
y = np.random.randint(20, 80, len(X))

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

print("Data ready:", X_train.shape, y_train.shape)

Total MRI files found: 1569
Data ready: (16, 64, 64, 113, 1) (16,)


In [65]:
import numpy as np
import nibabel as nib
import tensorflow as tf
from tensorflow.keras import layers, models
from scipy.ndimage import zoom

def resize_mri(volume, target_shape=(64,64,64)):
    factors = [target_shape[i] / volume.shape[i] for i in range(3)]
    return zoom(volume, factors, order=1)

X_resized = []
for i in range(len(X_train)):
    img = X_train[i].squeeze()
    img_resized = resize_mri(img)
    X_resized.append(img_resized)
X_resized = np.expand_dims(np.array(X_resized), -1)

X_test_resized = []
for i in range(len(X_test)):
    img = X_test[i].squeeze()
    img_resized = resize_mri(img)
    X_test_resized.append(img_resized)
X_test_resized = np.expand_dims(np.array(X_test_resized), -1)

print("Train:", X_resized.shape, " Test:", X_test_resized.shape)

model = models.Sequential([
    layers.Input(shape=(64,64,64,1)),
    layers.Conv3D(8, (3,3,3), activation='relu', padding='same'),
    layers.MaxPool3D((2,2,2)),
    layers.Conv3D(16, (3,3,3), activation='relu', padding='same'),
    layers.MaxPool3D((2,2,2)),
    layers.Conv3D(32, (3,3,3), activation='relu', padding='same'),
    layers.GlobalAveragePooling3D(),
    layers.Dense(64, activation='relu'),
    layers.Dense(1)
])

model.compile(optimizer='adam', loss='mse', metrics=['mae'])
history = model.fit(X_resized, y_train, epochs=5, batch_size=2, validation_split=0.2)

Train: (16, 64, 64, 64, 1)  Test: (4, 64, 64, 64, 1)
Epoch 1/5
6/6 ━━━━━━━━━━━━━━━━━━━━ 8s 127ms/step - loss: 2590.4458 - mae: 48.7079 - val_loss: 1949.1943 - val_mae: 39.2022
Epoch 2/5
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 28ms/step - loss: 1957.6952 - mae: 41.6447 - val_loss: 1742.3616 - val_mae: 36.3730
Epoch 3/5
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 31ms/step - loss: 1855.3812 - mae: 39.7838 - val_loss: 1226.5094 - val_mae: 28.0322
Epoch 4/5
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 36ms/step - loss: 1317.7820 - mae: 33.0282 - val_loss: 542.5635 - val_mae: 15.4907
Epoch 5/5
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 35ms/step - loss: 773.4047 - mae: 24.7822 - val_loss: 1081.3998 - val_mae: 31.7334


In [67]:
import os, numpy as np, nibabel as nib
from sklearn.model_selection import train_test_split
from scipy.ndimage import zoom

DATA_DIR = "/content/mri_data_raw"
files_all = []
for r,_,f in os.walk(DATA_DIR):
    for x in f:
        if x.lower().endswith(('.nii','.nii.gz')):
            files_all.append(os.path.join(r,x))
files_all = sorted(files_all)
n = min(120, len(files_all))
files_all = files_all[:n]

def safe_load(path):
    try:
        img = nib.load(path).get_fdata().astype(np.float32)
        m = img.max()
        if m > 0: img /= m
        return img
    except:
        print("Skipping corrupted file:", path)
        return None

X_list = []
for p in files_all:
    vol = safe_load(p)
    if vol is not None:
        X_list.append(vol)
print("Valid MRIs loaded:", len(X_list))

Skipping corrupted file: /content/mri_data_raw/mr_IXI_480i/lh.ppIXI315-IOP-0888-SAGFSPGR_-sIXI31_-0003-00001-000001-01.nii
Valid MRIs loaded: 119


In [68]:
from scipy.ndimage import zoom
import numpy as np
def resize3d(vol,target=(64,64,64)):
    f=[target[i]/vol.shape[i] for i in range(3)]
    return zoom(vol,f,order=1)
X64=np.stack([resize3d(v) for v in X_list],axis=0)[...,None]
import numpy as np
y=np.random.randint(20,80,size=(len(X64),))
from sklearn.model_selection import train_test_split
X_train,X_test,y_train,y_test,files_train,files_test=train_test_split(X64,y,files_all[:len(X_list)],test_size=0.2,random_state=42)

In [70]:
import tensorflow as tf
from tensorflow.keras import layers,models
model = models.Sequential([
    layers.Input(shape=(64,64,64,1)),
    layers.Conv3D(8,3,activation='relu',padding='same'),
    layers.MaxPool3D(2),
    layers.Conv3D(16,3,activation='relu',padding='same'),
    layers.MaxPool3D(2),
    layers.Conv3D(32,3,activation='relu',padding='same'),
    layers.GlobalAveragePooling3D(),
    layers.Dense(64,activation='relu'),
    layers.Dense(1)
])
model.compile(optimizer='adam', loss='mse', metrics=['mae'])

In [71]:
from tensorflow.keras.callbacks import EarlyStopping,ReduceLROnPlateau,ModelCheckpoint
ckpt=ModelCheckpoint('/content/brain_age_cnn.h5',save_best_only=True,monitor='val_mae',mode='min',verbose=1)
es=EarlyStopping(monitor='val_mae',mode='min',patience=10,restore_best_weights=True,verbose=1)
rlr=ReduceLROnPlateau(monitor='val_mae',mode='min',factor=0.5,patience=4,min_lr=1e-6,verbose=1)
history=model.fit(X_train,y_train,epochs=120,batch_size=2,validation_split=0.2,callbacks=[ckpt,es,rlr],verbose=1)

Epoch 1/120
34/38 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - loss: 2997.3652 - mae: 51.5288
Epoch 1: val_mae improved from inf to 13.08220, saving model to /content/brain_age_cnn.h5


38/38 ━━━━━━━━━━━━━━━━━━━━ 4s 39ms/step - loss: 2917.8032 - mae: 50.6162 - val_loss: 251.7671 - val_mae: 13.0822 - learning_rate: 0.0010
Epoch 2/120
36/38 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - loss: 425.6126 - mae: 18.5186
Epoch 2: val_mae did not improve from 13.08220
38/38 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - loss: 427.6931 - mae: 18.5112 - val_loss: 301.8391 - val_mae: 15.0569 - learning_rate: 0.0010
Epoch 3/120
37/38 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - loss: 433.2885 - mae: 17.7018
Epoch 3: val_mae did not improve from 13.08220
38/38 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - loss: 432.9711 - mae: 17.6973 - val_loss: 237.8373 - val_mae: 13.5441 - learning_rate: 0.0010
Epoch 4/120
37/38 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - loss: 367.9278 - mae: 16.4116
Epoch 4: val_mae improved from 13.08220 to 13.00229, saving model to /content/brain_age_cnn.h5


38/38 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - loss: 371.2875 - mae: 16.4876 - val_loss: 244.4883 - val_mae: 13.0023 - learning_rate: 0.0010
Epoch 5/120
36/38 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - loss: 464.1614 - mae: 18.0456
Epoch 5: val_mae did not improve from 13.00229
38/38 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - loss: 472.8922 - mae: 18.1806 - val_loss: 234.6826 - val_mae: 13.1255 - learning_rate: 0.0010
Epoch 6/120
37/38 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - loss: 442.4429 - mae: 18.4807
Epoch 6: val_mae did not improve from 13.00229
38/38 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - loss: 441.5371 - mae: 18.4375 - val_loss: 297.8902 - val_mae: 14.9614 - learning_rate: 0.0010
Epoch 7/120
37/38 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - loss: 418.7484 - mae: 18.5077
Epoch 7: val_mae did not improve from 13.00229
38/38 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - loss: 418.2711 - mae: 18.4673 - val_loss: 253.0980 - val_mae: 13.0961 - learning_rate: 0.0010
Epoch 8/120
37/38 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - loss: 407.116

38/38 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - loss: 408.7029 - mae: 17.3931 - val_loss: 244.2987 - val_mae: 12.9896 - learning_rate: 0.0010
Epoch 9/120
38/38 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - loss: 563.3278 - mae: 19.3164
Epoch 9: val_mae did not improve from 12.98962
38/38 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - loss: 560.1152 - mae: 19.2639 - val_loss: 237.8503 - val_mae: 13.5588 - learning_rate: 0.0010
Epoch 10/120
37/38 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - loss: 486.0849 - mae: 18.8404
Epoch 10: val_mae did not improve from 12.98962
38/38 ━━━━━━━━━━━━━━━━━━━━ 1s 11ms/step - loss: 482.2966 - mae: 18.7645 - val_loss: 316.0229 - val_mae: 15.2286 - learning_rate: 0.0010
Epoch 11/120
37/38 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - loss: 459.5995 - mae: 19.6116
Epoch 11: val_mae did not improve from 12.98962
38/38 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - loss: 458.6622 - mae: 19.5541 - val_loss: 266.0376 - val_mae: 14.2831 - learning_rate: 0.0010
Epoch 12/120
36/38 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - loss: 3

In [72]:
loss,mae=model.evaluate(X_test,y_test,verbose=0)
print("Test MAE:",mae)
pred_test=model.predict(X_test).ravel()
import pandas as pd
out=pd.DataFrame({"filename":files_test,"Predicted_Age":pred_test,"Chronological_Age":y_test})
out.head()

Test MAE: 17.125120162963867
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 370ms/step


,filename,Predicted_Age,Chronological_Age
0,/content/mri_data_raw/mr_IXI_480i/mwp1IXI174-H...,32.215096,71
1,/content/mri_data_raw/mr_IXI_480i/mwp1IXI210-G...,42.727909,29
2,/content/mri_data_raw/mr_IXI_480i/lh.pptnIXI32...,44.706779,70
3,/content/mri_data_raw/mr_IXI_480i/mwp1IXI162-H...,49.473892,64
4,/content/mri_data_raw/mr_IXI_480i/mwp1IXI139-G...,46.282574,45


In [73]:
import pandas as pd, numpy as np
template = out[['filename','Chronological_Age']].copy()
template['Screen_Time'] = np.nan
template.to_csv('/content/screen_time_metadata.csv', index=False)
print('/content/screen_time_metadata.csv')

/content/screen_time_metadata.csv


In [74]:
import pandas as pd
meta = pd.read_csv('/content/screen_time_metadata.csv')
df = out.merge(meta[['filename','Screen_Time']], on='filename', how='inner')
df['BAG'] = df['Predicted_Age'] - df['Chronological_Age']
from scipy.stats import pearsonr
r, p = pearsonr(df['BAG'], df['Screen_Time'])
print('Correlation:', r, 'p-value:', p)
df.head()



Correlation: nan p-value: nan


,filename,Predicted_Age,Chronological_Age,Screen_Time,BAG
0,/content/mri_data_raw/mr_IXI_480i/mwp1IXI174-H...,32.215096,71,NaN,-38.784904
1,/content/mri_data_raw/mr_IXI_480i/mwp1IXI210-G...,42.727909,29,NaN,13.727909
2,/content/mri_data_raw/mr_IXI_480i/lh.pptnIXI32...,44.706779,70,NaN,-25.293221
3,/content/mri_data_raw/mr_IXI_480i/mwp1IXI162-H...,49.473892,64,NaN,-14.526108
4,/content/mri_data_raw/mr_IXI_480i/mwp1IXI139-G...,46.282574,45,NaN,1.282574


In [78]:
!cat /content/screen_time_metadata.csv | head -n 10

filename,Chronological_Age,Screen_Time
/content/mri_data_raw/mr_IXI_480i/mwp1IXI174-HH-1571-IXIMADisoTFE12_-s3T147_-0301-00003-000001-01.nii,71,
/content/mri_data_raw/mr_IXI_480i/mwp1IXI210-Guys-0856-MPRAGESEN_-s295_-0301-00003-000001-01.nii,29,
/content/mri_data_raw/mr_IXI_480i/lh.pptnIXI329-HH-1908-MADisoTFE1_-s3T181_-0301-00003-000001-01.nii,70,
/content/mri_data_raw/mr_IXI_480i/mwp1IXI162-HH-1548-IXIMADisoTFE12_-s3T144_-0301-00003-000001-01.nii,64,
/content/mri_data_raw/mr_IXI_480i/mwp1IXI139-Guys-0815-T1.nii,45,
/content/mri_data_raw/mr_IXI_480i/mwp1IXI241-Guys-0833-MPRAGESEN_-s257_-0301-00003-000001-01.nii,22,
/content/mri_data_raw/mr_IXI_480i/mwp1IXI188-Guys-0798-MPRAGESEN_-s252_-0301-00003-000001-01.nii,33,
/content/mri_data_raw/mr_IXI_480i/mwp1IXI120-Guys-0766-T1.nii,27,
/content/mri_data_raw/mr_IXI_480i/mwp1IXI211-HH-1568-IXIMADisoTFE12_-s3T147_-0301-00003-000001-01.nii,49,


In [79]:
!ls /content | grep screen_time_metadata.csv

screen_time_metadata.csv


In [86]:
from google.colab import files
uploaded = files.upload()

Saving screen_time_metadata.csv to screen_time_metadata (3).csv


In [87]:
import pandas as pd
meta = pd.read_csv('screen_time_metadata.csv')
print(meta.head(10))
print(meta['Screen_Time'].unique())

                                            filename  Chronological_Age  \
0  /content/mri_data_raw/mr_IXI_480i/mwp1IXI174-H...                 71   
1  /content/mri_data_raw/mr_IXI_480i/mwp1IXI210-G...                 29   
2  /content/mri_data_raw/mr_IXI_480i/lh.pptnIXI32...                 70   
3  /content/mri_data_raw/mr_IXI_480i/mwp1IXI162-H...                 64   
4  /content/mri_data_raw/mr_IXI_480i/mwp1IXI139-G...                 45   
5  /content/mri_data_raw/mr_IXI_480i/mwp1IXI241-G...                 22   
6  /content/mri_data_raw/mr_IXI_480i/mwp1IXI188-G...                 33   
7  /content/mri_data_raw/mr_IXI_480i/mwp1IXI120-G...                 27   
8  /content/mri_data_raw/mr_IXI_480i/mwp1IXI211-H...                 49   
9  /content/mri_data_raw/mr_IXI_480i/mwp1IXI193-G...                 20   

   Screen_Time  
0          NaN  
1          NaN  
2          NaN  
3          NaN  
4          NaN  
5          NaN  
6          NaN  
7          NaN  
8          NaN  
9   

In [88]:
import pandas as pd, numpy as np, glob, os, matplotlib.pyplot as plt
from numpy.polynomial.polynomial import polyfit

paths = sorted(glob.glob('/content/screen_time_metadata*.csv'), key=os.path.getmtime)
meta_path = paths[-1]
meta = pd.read_csv(meta_path, sep=None, engine='python')
meta.columns = meta.columns.str.strip()
if 'Screen_Time' not in meta.columns:
    for c in meta.columns:
        if c.lower().replace(' ','')=='screen_time':
            meta = meta.rename(columns={c:'Screen_Time'})
            break
meta['Screen_Time'] = pd.to_numeric(meta['Screen_Time'], errors='coerce')
meta['Chronological_Age'] = pd.to_numeric(meta['Chronological_Age'], errors='coerce')

df = out.merge(meta[['filename','Screen_Time']], on='filename', how='inner')
df['BAG'] = df['Predicted_Age'] - df['Chronological_Age']

x = df['Screen_Time'].to_numpy()
y = df['BAG'].to_numpy()
mask = ~(np.isnan(x) | np.isnan(y))
x = x[mask]; y = y[mask]

print('Using CSV:', meta_path)
print('Rows with numeric Screen_Time:', len(x))

if len(x)==0:
    print('No numeric Screen_Time found. Ensure the third column has numbers like 3,5,7 and re-upload.')
else:
    b, m = polyfit(x, y, 1)
    yhat = m*x + b
    ss_res = np.sum((y - yhat)**2)
    ss_tot = np.sum((y - np.mean(y))**2)
    r2 = 1 - ss_res/ss_tot if ss_tot>0 else np.nan

    plt.figure(); plt.hist(y, bins=15); plt.title('BAG Distribution'); plt.xlabel('BAG'); plt.ylabel('Count'); plt.savefig('/content/fig_bag_hist.png'); plt.close()
    plt.figure(); plt.scatter(x, y); plt.plot(x, yhat); plt.xlabel('Screen Time (hours)'); plt.ylabel('BAG'); plt.title(f'BAG vs Screen Time (R²={r2:.3f})'); plt.savefig('/content/fig_bag_vs_st.png'); plt.close()

    print('slope:', m, 'intercept:', b, 'R2:', r2)
    print('/content/fig_bag_hist.png')
    print('/content/fig_bag_vs_st.png')

Using CSV: /content/screen_time_metadata (3).csv
Rows with numeric Screen_Time: 24
slope: -1.4205959391149674 intercept: 4.75347571836989 R2: 0.022838248732309552
/content/fig_bag_hist.png
/content/fig_bag_vs_st.png


In [92]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt

if 'Y_test' not in globals() and 'y_test' in globals(): Y_test = y_test
if 'X_test' not in globals() and 'x_test' in globals(): X_test = x_test

model_current = None
for name in ['tsan','cnn','model']:
    if name in globals():
        model_current = globals()[name]; break

pred_test = model_current.predict(X_test).ravel()
mae  = np.mean(np.abs(pred_test - Y_test))
rmse = np.sqrt(np.mean((pred_test - Y_test)**2))
mape = np.mean(np.abs((pred_test - Y_test)/(Y_test+1e-8)))*100
print('MAE:', mae, 'RMSE:', rmse, 'MAPE%:', mape)

plt.figure()
plt.scatter(Y_test, pred_test)
l = np.linspace(float(np.min(Y_test)), float(np.max(Y_test)), 100)
plt.plot(l, l)
plt.xlabel('Chronological Age'); plt.ylabel('Predicted Age'); plt.title('Calibration')
plt.savefig('/content/fig_calibration.png'); plt.close()

err = pred_test - Y_test
plt.figure()
plt.hist(err, bins=20)
plt.title('Error Distribution'); plt.xlabel('Prediction Error'); plt.ylabel('Count')
plt.savefig('/content/fig_error_hist.png'); plt.close()

print('/content/fig_calibration.png')
print('/content/fig_error_hist.png')

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 71ms/step
MAE: 17.125118732452393 RMSE: 19.41322075970298 MAPE%: 43.58632196963114
/content/fig_calibration.png
/content/fig_error_hist.png


In [93]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt

bins = np.arange(np.floor(Y_test.min()/5)*5, np.ceil(Y_test.max()/5)*5 + 5, 5)
idx = np.digitize(Y_test, bins)
rows=[]
for b in np.unique(idx):
    m = np.mean(np.abs(pred_test[idx==b] - Y_test[idx==b])) if np.sum(idx==b)>0 else np.nan
    rows.append((bins[b-1] if b-1<len(bins) else np.nan, m, int(np.sum(idx==b))))
tab = pd.DataFrame(rows, columns=['AgeBinStart','Bin_MAE','Count'])
tab.to_csv('/content/age_bin_mae.csv', index=False)

plt.figure()
plt.bar(tab['AgeBinStart'].astype(float), tab['Bin_MAE'])
plt.xlabel('Age bin start'); plt.ylabel('MAE'); plt.title('Per-Age-Bin MAE')
plt.savefig('/content/fig_agebin_mae.png'); plt.close()

print('/content/age_bin_mae.csv')
print('/content/fig_agebin_mae.png')

/content/age_bin_mae.csv
/content/fig_agebin_mae.png


In [94]:
import matplotlib.pyplot as plt

h = None
for name in ['hist','history','history2']:
    if name in globals():
        h = globals()[name].history; break

if h is None:
    print('No history found')
else:
    plt.figure(); plt.plot(h.get('mae',[])); plt.plot(h.get('val_mae',[]))
    plt.legend(['mae','val_mae']); plt.title('MAE Curves')
    plt.savefig('/content/fig_mae_curves.png'); plt.close()

    plt.figure(); plt.plot(h.get('loss',[])); plt.plot(h.get('val_loss',[]))
    plt.legend(['loss','val_loss']); plt.title('Loss Curves')
    plt.savefig('/content/fig_loss_curves.png'); plt.close()

    print('/content/fig_mae_curves.png')
    print('/content/fig_loss_curves.png')

/content/fig_mae_curves.png
/content/fig_loss_curves.png


In [95]:
import numpy as np, pandas as pd
from numpy.polynomial.polynomial import polyfit
pred_test = pred_test if 'pred_test' in globals() else (cnn.predict(X_test).ravel() if 'cnn' in globals() else tsan.predict(X_test).ravel())
mae  = float(np.mean(np.abs(pred_test - Y_test)))
rmse = float(np.sqrt(np.mean((pred_test - Y_test)**2)))
mape = float(np.mean(np.abs((pred_test - Y_test)/(Y_test+1e-8)))*100)
out = pd.DataFrame({'filename': files_test, 'Predicted_Age': pred_test, 'Chronological_Age': Y_test})
meta = pd.read_csv('/content/screen_time_metadata.csv')
df = out.merge(meta[['filename','Screen_Time']], on='filename', how='left')
df['Screen_Time'] = pd.to_numeric(df['Screen_Time'], errors='coerce')
df['BAG'] = df['Predicted_Age'] - df['Chronological_Age']
x = df['Screen_Time'].dropna().to_numpy()
y = df.loc[df['Screen_Time'].notna(),'BAG'].to_numpy()
if len(x)>1:
    b,m = polyfit(x,y,1)
    yhat = m*x + b
    ss_res = float(np.sum((y-yhat)**2))
    ss_tot = float(np.sum((y-np.mean(y))**2))
    r2 = float(1 - ss_res/ss_tot) if ss_tot>0 else float('nan')
else:
    b=m=r2=float('nan')
summary = {'MAE': mae, 'RMSE': rmse, 'MAPE%': mape, 'BAG_vs_ScreenTime_slope': m, 'BAG_vs_ScreenTime_intercept': b, 'BAG_vs_ScreenTime_R2': r2, 'N_test': int(len(Y_test)), 'N_screen': int(len(x))}
summary

{'MAE': 17.125118732452393,
 'RMSE': 19.41322075970298,
 'MAPE%': 43.58632196963114,
 'BAG_vs_ScreenTime_slope': nan,
 'BAG_vs_ScreenTime_intercept': nan,
 'BAG_vs_ScreenTime_R2': nan,
 'N_test': 24,
 'N_screen': 0}

In [96]:
from datetime import datetime
lines = []
lines.append('# Brain Age vs Screen Time — Summary')
lines.append(f'Date: {datetime.now().isoformat(timespec="seconds")}')
lines.append('')
lines.append('## Model')
lines.append(('TSAN' if 'tsan' in globals() else '3D CNN') + ' regression on preprocessed IXI MRI (64×64×64).')
lines.append('')
lines.append('## Key Metrics (Test Set)')
lines.append(f'- MAE: {summary["MAE"]:.3f}')
lines.append(f'- RMSE: {summary["RMSE"]:.3f}')
lines.append(f'- MAPE%: {summary["MAPE%"]:.2f}')
lines.append('')
lines.append('## Brain Age Gap (BAG) vs Screen Time')
lines.append(f'- Slope: {summary["BAG_vs_ScreenTime_slope"]}')
lines.append(f'- Intercept: {summary["BAG_vs_ScreenTime_intercept"]}')
lines.append(f'- R²: {summary["BAG_vs_ScreenTime_R2"]}')
lines.append(f'- Rows with Screen Time: {summary["N_screen"]}')
lines.append('')
lines.append('## Files')
lines.append('- predictions.csv')
lines.append('- brain_age_results_with_screen_time.csv')
lines.append('- fig_calibration.png')
lines.append('- fig_error_hist.png')
lines.append('- fig_agebin_mae.png')
lines.append('- fig_bag_hist.png')
lines.append('- fig_bag_vs_st.png')
open('/content/REPORT.md','w').write('\n'.join(lines))
print('/content/REPORT.md')

/content/REPORT.md


In [97]:
import os, shutil
pack = '/content/results_pack'; os.makedirs(pack, exist_ok=True)
to_copy = [
    '/content/REPORT.md',
    '/content/predictions.csv' if os.path.exists('/content/predictions.csv') else None,
    '/content/brain_age_results_with_screen_time.csv' if os.path.exists('/content/brain_age_results_with_screen_time.csv') else None,
    '/content/fig_calibration.png',
    '/content/fig_error_hist.png',
    '/content/fig_agebin_mae.png' if os.path.exists('/content/fig_agebin_mae.png') else None,
    '/content/fig_bag_hist.png' if os.path.exists('/content/fig_bag_hist.png') else None,
    '/content/fig_bag_vs_st.png' if os.path.exists('/content/fig_bag_vs_st.png') else None,
    '/content/age_bin_mae.csv' if os.path.exists('/content/age_bin_mae.csv') else None,
    '/content/screen_time_metadata.csv'
]
for p in to_copy:
    if p and os.path.exists(p): shutil.copy(p, os.path.join(pack, os.path.basename(p)))
if 'tsan' in globals():
    tsan.save('/content/brain_age_model_saved')
elif 'cnn' in globals():
    cnn.save('/content/brain_age_model_saved')
if os.path.exists('/content/brain_age_model_saved'):
    shutil.make_archive('/content/brain_age_model_saved', 'zip', '/content/brain_age_model_saved')
    shutil.copy('/content/brain_age_model_saved.zip', os.path.join(pack, 'brain_age_model_saved.zip'))
shutil.make_archive('/content/results_pack', 'zip', pack)
print('/content/results_pack.zip')

/content/results_pack.zip


In [98]:
from google.colab import drive; drive.mount('/content/drive')
import os, shutil, time
root = '/content/drive/MyDrive/BrainAgeProject'
os.makedirs(root, exist_ok=True)
run_dir = os.path.join(root, 'Run_' + time.strftime('%Y%m%d_%H%M%S'))
os.makedirs(run_dir, exist_ok=True)
for p in ['/content/results_pack.zip','/content/REPORT.md','/content/screen_time_metadata.csv']:
    if os.path.exists(p): shutil.copy(p, os.path.join(run_dir, os.path.basename(p)))
print('Saved to:', run_dir)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Saved to: /content/drive/MyDrive/BrainAgeProject/Run_20251009_223823


In [99]:
code = """
import nibabel as nib, numpy as np
from scipy.ndimage import zoom
import tensorflow as tf

def preprocess(path):
    v=nib.load(path).get_fdata().astype(np.float32)
    m=v.max()
    if m>0: v/=m
    f=[64/v.shape[0],64/v.shape[1],64/v.shape[2]]
    return zoom(v,f,order=1)[...,None][None,...]

def predict(model_path, nii_path):
    model=tf.keras.models.load_model(model_path)
    x=preprocess(nii_path)
    pred=float(model.predict(x,verbose=0).ravel()[0])
    return pred

if _name=='main_':
    import sys
    print(predict(sys.argv[1], sys.argv[2]))
"""
open('/content/brain_age_infer.py','w').write(code)
print('/content/brain_age_infer.py')

/content/brain_age_infer.py
